In [1]:
import pandas as pd

In [ ]:
df = pd.read_csv("open_cravat_curation_v2_imputed.csv")

print(df.head(0))

In [ ]:
with open("columns.txt", "w") as f:
    f.write(", ".join(df.loc[1].to_string().split(", ")))

In [ ]:
df.drop(columns=["clinvar__sig", "clinvar__id", "Germline review status", "Stars"], axis=1, inplace=True)

In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# 1. Veriyi yüklediğini varsayıyoruz
# df = pd.read_csv("verisetin.csv")

# 2. Boolean (True/False) değerleri 1 ve 0'a çevirme
bool_cols = ['polarity_change', 'chirality_shift']
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

# 3. Sekans Sütunlarını Sayısallaştırma (K-mer Extraction)
seq_columns = ['Ref_Sequence', 'Alt_Sequence', 'DNA_11mer_Ref', 'DNA_11mer_Alt', 'Prot_11mer_Ref', 'Prot_11mer_Alt']

# 3'lü karakterler (trimer/kodon) halinde sayacağız
vectorizer = CountVectorizer(analyzer='char', ngram_range=(3, 3)) 

seq_features = [] # Oluşturulan yeni sayısal sütunları tutacağımız liste

for col in seq_columns:
    if col in df.columns:        
        # Sekansları k-mer frekans matrisine dönüştürme
        seq_matrix = vectorizer.fit_transform(df[col])
        
        # Yeni sütun isimlerini oluşturma (örn: Ref_Sequence_ATG)
        feature_names = [f"{col}_{motif}" for motif in vectorizer.get_feature_names_out()]
        
        # Matrisi DataFrame'e çevirip listeye ekleme
        seq_df = pd.DataFrame(seq_matrix.toarray(), columns=feature_names, index=df.index)
        seq_features.append(seq_df)

# Orjinal sekans metin sütunlarını düşürüp, k-mer sayısal sütunlarını ana veriye ekliyoruz
df = df.drop(columns=seq_columns, errors='ignore')
df = pd.concat([df] + seq_features, axis=1)

# 4. Geriye Kalan Kategorik Verileri (Gen isimleri vs.) LightGBM formatına getirme
object_cols = df.select_dtypes(include=['object']).columns.tolist()
for col in object_cols:
    df[col] = df[col].astype('category')

# 5. X ve y'yi ayırma
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 6. Modeli Kurma ve Eğitme
model = lgb.LGBMClassifier(objective='binary', random_state=42, class_weight='balanced')

model.fit(
    X_train, 
    y_train,
    eval_set=[(X_test, y_test)],
    categorical_feature=object_cols
)

print("Eğitim tamamlandı! Sekans verileri k-mer frekansları olarak modele dahil edildi.")

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, f1_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

best_models_dict = {}
reports_dict = {}

panels = df['Panel'].unique()
print(f"Toplam {len(panels)} panel bulundu. Optuna ile kapsamlı optimizasyon başlıyor...\n")

for panel_name in panels:
    print(f"\n{'='*60}")
    print(f">>> Panel: {panel_name} için en iyi parametreler aranıyor...")
    
    panel_df = df[df['Panel'] == panel_name].copy()
    
    if panel_df['target'].nunique() < 2:
        print(f"UYARI: {panel_name} panelinde sadece tek sınıf var! Atlanıyor.")
        continue
        
    X = panel_df.drop(columns=['target', 'Panel'])
    y = panel_df['target']
    
    categorical_features = X.select_dtypes(include=['category', 'object']).columns.tolist()
    for col in categorical_features:
        X[col] = X[col].astype('category')
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    # --- Optuna objective fonksiyonu ---
    def objective(trial):
        params = {
            'objective': 'binary',
            'verbosity': -1,
            'random_state': 42,
            'class_weight': 'balanced',
            
            # Ağaç yapısı
            'num_leaves': trial.suggest_int('num_leaves', 10, 120),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'min_child_samples': trial.suggest_int('min_child_samples', 3, 50),
            'min_child_weight': trial.suggest_float('min_child_weight', 1e-5, 10.0, log=True),
            
            # Öğrenme hızı ve iterasyon
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 50, 800),
            
            # Regularizasyon
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            
            # Örnekleme (overfitting'e karşı)
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
            'subsample': trial.suggest_float('subsample', 0.3, 1.0),
            'subsample_freq': trial.suggest_int('subsample_freq', 1, 7),
            
            # Ekstra
            'max_bin': trial.suggest_int('max_bin', 63, 511),
            'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
        }
        
        # 5-Fold Stratified Cross Validation ile F1 hesapla
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        f1_scores = []
        
        for train_idx, val_idx in skf.split(X_train, y_train):
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            model = lgb.LGBMClassifier(**params)
            model.fit(
                X_tr, y_tr,
                categorical_feature=categorical_features
            )
            
            y_val_pred = model.predict(X_val)
            f1_scores.append(f1_score(y_val, y_val_pred))
        
        return np.mean(f1_scores)
    
    # --- Optuna çalıştır: 200 deneme ---
    study = optuna.create_study(direction='maximize', study_name=panel_name)
    study.optimize(objective, n_trials=200, show_progress_bar=True)
    
    print(f"En iyi CV F1 skoru: {study.best_value:.4f}")
    print(f"En iyi parametreler: {study.best_params}")
    
    # En iyi parametrelerle final modeli eğit (tüm eğitim verisi üzerinde)
    best_params = study.best_params
    best_params.update({
        'objective': 'binary',
        'verbosity': -1,
        'random_state': 42,
        'class_weight': 'balanced',
    })
    
    best_model = lgb.LGBMClassifier(**best_params)
    best_model.fit(X_train, y_train, categorical_feature=categorical_features)
    
    # Test seti üzerinde değerlendir
    y_pred = best_model.predict(X_test)
    y_pred_proba = best_model.predict_proba(X_test)[:, 1]
    
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"\n--- {panel_name} Test Performansı (Optuna Optimize) ---")
    print(f"ROC-AUC Skoru: {roc_auc:.4f}")
    print(f"F1 Skoru (sınıf 1): {f1_score(y_test, y_pred):.4f}\n")
    print(classification_report(y_test, y_pred))
    
    # Threshold optimizasyonu: varsayılan 0.5 yerine en iyi F1 veren eşiği bul
    thresholds = np.arange(0.20, 0.80, 0.01)
    best_thr, best_thr_f1 = 0.5, 0.0
    for thr in thresholds:
        y_thr = (y_pred_proba >= thr).astype(int)
        thr_f1 = f1_score(y_test, y_thr)
        if thr_f1 > best_thr_f1:
            best_thr_f1 = thr_f1
            best_thr = thr
    
    y_pred_optimized = (y_pred_proba >= best_thr).astype(int)
    print(f"--- Threshold Optimizasyonu (eşik={best_thr:.2f}) ---")
    print(f"F1 Skoru (optimize edilmiş eşik): {best_thr_f1:.4f}\n")
    print(classification_report(y_test, y_pred_optimized))
    
    best_models_dict[panel_name] = {
        'model': best_model,
        'best_params': best_params,
        'best_threshold': best_thr,
        'study': study,
    }

print("\n" + "="*60)
print("Tüm paneller için Optuna optimizasyonu tamamlandı!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import f1_score, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay

fig_count = len([p for p in best_models_dict])
fig, axes = plt.subplots(fig_count, 4, figsize=(24, 6 * fig_count))
if fig_count == 1:
    axes = axes.reshape(1, -1)

for idx, (panel_name, info) in enumerate(best_models_dict.items()):
    model = info['model']
    study = info['study']
    best_thr = info['best_threshold']
    
    # Test verisini yeniden hazırla
    panel_df = df[df['Panel'] == panel_name].copy()
    X = panel_df.drop(columns=['target', 'Panel'])
    y = panel_df['target']
    categorical_features = X.select_dtypes(include=['category', 'object']).columns.tolist()
    for col in categorical_features:
        X[col] = X[col].astype('category')
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred_opt = (y_pred_proba >= best_thr).astype(int)
    
    # 1) Optuna optimizasyon süreci
    ax1 = axes[idx, 0]
    trials = study.trials
    values = [t.value for t in trials]
    best_so_far = np.maximum.accumulate(values)
    ax1.plot(values, alpha=0.3, color='steelblue', label='Her deneme')
    ax1.plot(best_so_far, color='red', linewidth=2, label='En iyi (kümülatif)')
    ax1.set_title(f'{panel_name}\nOptuna Optimizasyon Süreci', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Deneme No')
    ax1.set_ylabel('F1 Skoru (CV)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2) ROC Eğrisi
    ax2 = axes[idx, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc_val = auc(fpr, tpr)
    ax2.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'ROC (AUC = {roc_auc_val:.4f})')
    ax2.plot([0, 1], [0, 1], color='gray', linestyle='--', alpha=0.5)
    ax2.set_title(f'{panel_name}\nROC Eğrisi', fontsize=12, fontweight='bold')
    ax2.set_xlabel('False Positive Rate')
    ax2.set_ylabel('True Positive Rate')
    ax2.legend(loc='lower right')
    ax2.grid(True, alpha=0.3)
    
    # 3) Confusion Matrix (optimize edilmiş threshold ile)
    ax3 = axes[idx, 2]
    cm = confusion_matrix(y_test, y_pred_opt)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Benign (0)', 'Pathogenic (1)'])
    disp.plot(ax=ax3, cmap='Blues', colorbar=False)
    ax3.set_title(f'{panel_name}\nConfusion Matrix (eşik={best_thr:.2f})', fontsize=12, fontweight='bold')
    
    # 4) Threshold vs F1 grafiği
    ax4 = axes[idx, 3]
    thresholds = np.arange(0.10, 0.90, 0.01)
    f1_vals = [f1_score(y_test, (y_pred_proba >= t).astype(int), zero_division=0) for t in thresholds]
    ax4.plot(thresholds, f1_vals, color='green', linewidth=2)
    ax4.axvline(x=best_thr, color='red', linestyle='--', linewidth=1.5, label=f'En iyi eşik = {best_thr:.2f}')
    ax4.axvline(x=0.5, color='gray', linestyle=':', linewidth=1, label='Varsayılan (0.50)')
    ax4.set_title(f'{panel_name}\nThreshold vs F1 Skoru', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Threshold')
    ax4.set_ylabel('F1 Skoru')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Feature Importance (her panel için ayrı) ---
for panel_name, info in best_models_dict.items():
    model = info['model']
    
    importance = model.feature_importances_
    feature_names = model.feature_name_
    sorted_idx = np.argsort(importance)[-20:]
    
    plt.figure(figsize=(10, 7))
    plt.barh(range(len(sorted_idx)), importance[sorted_idx], color='teal', alpha=0.8)
    plt.yticks(range(len(sorted_idx)), [feature_names[i] for i in sorted_idx])
    plt.title(f'{panel_name} — En Önemli 20 Özellik (Gain)', fontsize=13, fontweight='bold')
    plt.xlabel('Feature Importance')
    plt.tight_layout()
    plt.show()

# --- Özet tablo ---
print("\n" + "="*70)
print(f"{'Panel':<25} {'ROC-AUC':>10} {'F1 (thr=0.5)':>14} {'F1 (opt thr)':>14} {'Eşik':>8}")
print("="*70)
for panel_name, info in best_models_dict.items():
    model = info['model']
    best_thr = info['best_threshold']
    panel_df = df[df['Panel'] == panel_name].copy()
    X = panel_df.drop(columns=['target', 'Panel'])
    y = panel_df['target']
    for col in X.select_dtypes(include=['category', 'object']).columns:
        X[col] = X[col].astype('category')
    _, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    y_proba = model.predict_proba(X_test)[:, 1]
    f1_default = f1_score(y_test, (y_proba >= 0.5).astype(int))
    f1_opt = f1_score(y_test, (y_proba >= best_thr).astype(int))
    roc = roc_auc_score(y_test, y_proba)
    print(f"{panel_name:<25} {roc:>10.4f} {f1_default:>14.4f} {f1_opt:>14.4f} {best_thr:>8.2f}")
print("="*70)


In [ ]:
# =============================================================================
# FEATURE ENGINEERING PIPELINE — Adım 1: Veri Hazırlık & Biyoinformatik Özellikler
# =============================================================================
import pandas as pd
import numpy as np

# --- 1. Veriyi temiz olarak yeniden yükle ---
df_fe = pd.read_csv("open_cravat_curation_v2_imputed.csv")
df_fe.drop(columns=["clinvar__sig", "clinvar__id", "Germline review status", "Stars"], inplace=True)

# Boolean → int
for col in ['polarity_change', 'chirality_shift']:
    if col in df_fe.columns:
        df_fe[col] = df_fe[col].astype(int)

# --- 2. Sekans sütunlarını DÜŞÜR (ESM1b ve AlphaMissense zaten sekans bağlamını kodluyor) ---
seq_columns = ['Ref_Sequence', 'Alt_Sequence', 'DNA_11mer_Ref', 'DNA_11mer_Alt',
               'Prot_11mer_Ref', 'Prot_11mer_Alt']

# Düşürmeden önce: DNA bağlam özelliklerini çıkar
if 'DNA_11mer_Ref' in df_fe.columns:
    # CpG bölgesi tespiti (mutasyon pozisyonu 6. karakter, 0-indexed=5)
    df_fe['is_cpg_site'] = df_fe['DNA_11mer_Ref'].apply(
        lambda s: int(s[5:7] == 'CG' or s[4:6] == 'CG') if isinstance(s, str) and len(s) >= 7 else 0
    )
    # GC content
    df_fe['gc_content_11mer'] = df_fe['DNA_11mer_Ref'].apply(
        lambda s: (s.count('G') + s.count('C')) / len(s) if isinstance(s, str) and len(s) > 0 else 0.5
    )

# Transition vs Transversion
if all(c in df_fe.columns for c in ['base__ref_base', 'base__alt_base']):
    transitions = {('A','G'), ('G','A'), ('C','T'), ('T','C')}
    df_fe['is_transition'] = df_fe.apply(
        lambda r: int((r['base__ref_base'], r['base__alt_base']) in transitions), axis=1
    )

# --- 3. Grantham Distance (amino asit substitüsyon şiddeti) ---
GRANTHAM = {
    ('A','R'):112,('A','N'):111,('A','D'):126,('A','C'):195,('A','Q'):91,('A','E'):107,
    ('A','G'):60,('A','H'):86,('A','I'):94,('A','L'):96,('A','K'):106,('A','M'):84,
    ('A','F'):113,('A','P'):27,('A','S'):99,('A','T'):58,('A','W'):148,('A','Y'):112,
    ('A','V'):64,('R','N'):86,('R','D'):96,('R','C'):180,('R','Q'):43,('R','E'):54,
    ('R','G'):125,('R','H'):29,('R','I'):97,('R','L'):102,('R','K'):26,('R','M'):91,
    ('R','F'):97,('R','P'):103,('R','S'):110,('R','T'):71,('R','W'):101,('R','Y'):77,
    ('R','V'):96,('N','D'):23,('N','C'):139,('N','Q'):46,('N','E'):42,('N','G'):80,
    ('N','H'):68,('N','I'):149,('N','L'):153,('N','K'):94,('N','M'):142,('N','F'):158,
    ('N','P'):91,('N','S'):46,('N','T'):65,('N','W'):174,('N','Y'):143,('N','V'):133,
    ('D','C'):154,('D','Q'):61,('D','E'):45,('D','G'):94,('D','H'):81,('D','I'):168,
    ('D','L'):172,('D','K'):101,('D','M'):160,('D','F'):177,('D','P'):108,('D','S'):65,
    ('D','T'):85,('D','W'):181,('D','Y'):160,('D','V'):152,('C','Q'):154,('C','E'):170,
    ('C','G'):159,('C','H'):174,('C','I'):198,('C','L'):198,('C','K'):202,('C','M'):196,
    ('C','F'):205,('C','P'):169,('C','S'):112,('C','T'):149,('C','W'):215,('C','Y'):194,
    ('C','V'):192,('Q','E'):29,('Q','G'):87,('Q','H'):24,('Q','I'):109,('Q','L'):113,
    ('Q','K'):53,('Q','M'):101,('Q','F'):116,('Q','P'):76,('Q','S'):68,('Q','T'):42,
    ('Q','W'):130,('Q','Y'):99,('Q','V'):96,('E','G'):98,('E','H'):40,('E','I'):134,
    ('E','L'):138,('E','K'):56,('E','M'):126,('E','F'):140,('E','P'):93,('E','S'):80,
    ('E','T'):65,('E','W'):152,('E','Y'):122,('E','V'):121,('G','H'):98,('G','I'):135,
    ('G','L'):138,('G','K'):127,('G','M'):127,('G','F'):153,('G','P'):42,('G','S'):56,
    ('G','T'):59,('G','W'):184,('G','Y'):147,('G','V'):109,('H','I'):94,('H','L'):99,
    ('H','K'):32,('H','M'):87,('H','F'):100,('H','P'):77,('H','S'):89,('H','T'):47,
    ('H','W'):115,('H','Y'):83,('H','V'):84,('I','L'):5,('I','K'):102,('I','M'):10,
    ('I','F'):21,('I','P'):95,('I','S'):142,('I','T'):89,('I','W'):61,('I','Y'):33,
    ('I','V'):29,('L','K'):107,('L','M'):15,('L','F'):22,('L','P'):98,('L','S'):145,
    ('L','T'):92,('L','W'):61,('L','Y'):36,('L','V'):32,('K','M'):95,('K','F'):102,
    ('K','P'):103,('K','S'):121,('K','T'):78,('K','W'):110,('K','Y'):85,('K','V'):97,
    ('M','F'):28,('M','P'):87,('M','S'):135,('M','T'):81,('M','W'):67,('M','Y'):36,
    ('M','V'):21,('F','P'):114,('F','S'):155,('F','T'):103,('F','W'):40,('F','Y'):22,
    ('F','V'):50,('P','S'):74,('P','T'):38,('P','W'):147,('P','Y'):110,('P','V'):68,
    ('S','T'):58,('S','W'):177,('S','Y'):144,('S','V'):124,('T','W'):128,('T','Y'):92,
    ('T','V'):69,('W','Y'):37,('W','V'):88,('Y','V'):55,
}
# Simetrik hale getir
grantham_full = {}
for (a, b), v in GRANTHAM.items():
    grantham_full[(a, b)] = v
    grantham_full[(b, a)] = v

def get_grantham(ref, alt):
    if not isinstance(ref, str) or not isinstance(alt, str) or len(ref) != 1 or len(alt) != 1:
        return np.nan
    if ref == alt:
        return 0
    return grantham_full.get((ref, alt), np.nan)

if all(c in df_fe.columns for c in ['ref_amino', 'alt_amino']):
    df_fe['grantham_distance'] = df_fe.apply(lambda r: get_grantham(r['ref_amino'], r['alt_amino']), axis=1)
    # Grantham kategorisi (ACMG uyumlu)
    df_fe['grantham_category'] = pd.cut(
        df_fe['grantham_distance'],
        bins=[-1, 50, 100, 150, 300],
        labels=[0, 1, 2, 3]  # conservative, mod_conservative, mod_radical, radical
    ).astype(float)

# --- 4. BLOSUM62 Skoru ---
BLOSUM62 = {
    ('A','A'):4,('A','R'):-1,('A','N'):-2,('A','D'):-2,('A','C'):0,('A','Q'):-1,('A','E'):-1,
    ('A','G'):0,('A','H'):-2,('A','I'):-1,('A','L'):-1,('A','K'):-1,('A','M'):-1,('A','F'):-2,
    ('A','P'):-1,('A','S'):1,('A','T'):0,('A','W'):-3,('A','Y'):-2,('A','V'):0,
    ('R','R'):5,('R','N'):0,('R','D'):-2,('R','C'):-3,('R','Q'):1,('R','E'):0,('R','G'):-2,
    ('R','H'):0,('R','I'):-3,('R','L'):-2,('R','K'):2,('R','M'):-1,('R','F'):-3,('R','P'):-2,
    ('R','S'):-1,('R','T'):-1,('R','W'):-3,('R','Y'):-2,('R','V'):-3,
    ('N','N'):6,('N','D'):1,('N','C'):-3,('N','Q'):0,('N','E'):0,('N','G'):0,('N','H'):1,
    ('N','I'):-3,('N','L'):-3,('N','K'):0,('N','M'):-2,('N','F'):-3,('N','P'):-2,('N','S'):1,
    ('N','T'):0,('N','W'):-4,('N','Y'):-2,('N','V'):-3,
    ('D','D'):6,('D','C'):-3,('D','Q'):0,('D','E'):2,('D','G'):-1,('D','H'):-1,('D','I'):-3,
    ('D','L'):-4,('D','K'):-1,('D','M'):-3,('D','F'):-3,('D','P'):-1,('D','S'):0,('D','T'):-1,
    ('D','W'):-4,('D','Y'):-3,('D','V'):-3,
    ('C','C'):9,('C','Q'):-3,('C','E'):-4,('C','G'):-3,('C','H'):-3,('C','I'):-1,('C','L'):-1,
    ('C','K'):-3,('C','M'):-1,('C','F'):-2,('C','P'):-3,('C','S'):-1,('C','T'):-1,('C','W'):-2,
    ('C','Y'):-2,('C','V'):-1,
    ('Q','Q'):5,('Q','E'):2,('Q','G'):-2,('Q','H'):0,('Q','I'):-3,('Q','L'):-2,('Q','K'):1,
    ('Q','M'):0,('Q','F'):-3,('Q','P'):-1,('Q','S'):0,('Q','T'):-1,('Q','W'):-2,('Q','Y'):-1,
    ('Q','V'):-2,
    ('E','E'):5,('E','G'):-2,('E','H'):0,('E','I'):-3,('E','L'):-3,('E','K'):1,('E','M'):-2,
    ('E','F'):-3,('E','P'):-1,('E','S'):0,('E','T'):-1,('E','W'):-3,('E','Y'):-2,('E','V'):-2,
    ('G','G'):6,('G','H'):-2,('G','I'):-4,('G','L'):-4,('G','K'):-2,('G','M'):-3,('G','F'):-3,
    ('G','P'):-2,('G','S'):0,('G','T'):-2,('G','W'):-2,('G','Y'):-3,('G','V'):-3,
    ('H','H'):8,('H','I'):-3,('H','L'):-3,('H','K'):-1,('H','M'):-2,('H','F'):-1,('H','P'):-2,
    ('H','S'):-1,('H','T'):-2,('H','W'):-2,('H','Y'):2,('H','V'):-3,
    ('I','I'):4,('I','L'):2,('I','K'):-3,('I','M'):1,('I','F'):0,('I','P'):-3,('I','S'):-2,
    ('I','T'):-1,('I','W'):-3,('I','Y'):-1,('I','V'):3,
    ('L','L'):4,('L','K'):-2,('L','M'):2,('L','F'):0,('L','P'):-3,('L','S'):-2,('L','T'):-1,
    ('L','W'):-2,('L','Y'):-1,('L','V'):1,
    ('K','K'):5,('K','M'):-1,('K','F'):-3,('K','P'):-1,('K','S'):0,('K','T'):-1,('K','W'):-3,
    ('K','Y'):-2,('K','V'):-2,
    ('M','M'):5,('M','F'):0,('M','P'):-2,('M','S'):-1,('M','T'):-1,('M','W'):-1,('M','Y'):-1,
    ('M','V'):1,
    ('F','F'):6,('F','P'):-4,('F','S'):-2,('F','T'):-2,('F','W'):1,('F','Y'):3,('F','V'):-1,
    ('P','P'):7,('P','S'):-1,('P','T'):-1,('P','W'):-4,('P','Y'):-3,('P','V'):-2,
    ('S','S'):4,('S','T'):1,('S','W'):-3,('S','Y'):-2,('S','V'):-2,
    ('T','T'):5,('T','W'):-2,('T','Y'):-2,('T','V'):0,
    ('W','W'):11,('W','Y'):2,('W','V'):-3,
    ('Y','Y'):7,('Y','V'):-1,
    ('V','V'):4,
}
blosum_full = {}
for (a, b), v in BLOSUM62.items():
    blosum_full[(a, b)] = v
    blosum_full[(b, a)] = v

def get_blosum62(ref, alt):
    if not isinstance(ref, str) or not isinstance(alt, str) or len(ref) != 1 or len(alt) != 1:
        return np.nan
    return blosum_full.get((ref, alt), np.nan)

if all(c in df_fe.columns for c in ['ref_amino', 'alt_amino']):
    df_fe['blosum62_score'] = df_fe.apply(lambda r: get_blosum62(r['ref_amino'], r['alt_amino']), axis=1)

# Sekans metin sütunlarını düşür (artık gerekli bilgileri çıkardık)
df_fe.drop(columns=[c for c in seq_columns if c in df_fe.columns], inplace=True)

# --- 5. gnomAD Türetilmiş Özellikler ---
if all(c in df_fe.columns for c in ['gnomad4__ac', 'gnomad4__an']):
    df_fe['gnomad4__af_computed'] = df_fe['gnomad4__ac'] / df_fe['gnomad4__an'].replace(0, np.nan)

if 'gnomad4__af' in df_fe.columns:
    df_fe['log10_af'] = np.log10(df_fe['gnomad4__af'].clip(lower=1e-8))
    df_fe['gnomad4__is_absent'] = (df_fe['gnomad4__af'] == 0).astype(int)
    # ACMG uyumlu frekans kategorileri
    df_fe['af_bin'] = pd.cut(
        df_fe['gnomad4__af'],
        bins=[-0.001, 0, 1e-5, 1e-3, 0.01, 1.0],
        labels=[0, 1, 2, 3, 4]  # absent, ultra_rare, rare, low_freq, common
    ).astype(float)

if all(c in df_fe.columns for c in ['gnomad4__nhomalt', 'gnomad4__ac']):
    df_fe['homozygote_ratio'] = df_fe['gnomad4__nhomalt'] / (df_fe['gnomad4__ac'] / 2).replace(0, np.nan)

# --- 6. Evrimsel korunum etkileşimi ---
if all(c in df_fe.columns for c in ['phylop__phylop100_vert', 'phastcons__phastcons100_vert']):
    df_fe['conservation_product'] = df_fe['phylop__phylop100_vert'] * df_fe['phastcons__phastcons100_vert']

# --- 7. CHASMplus: sadece ana skoru tut, kanser alt tiplerini düşür ---
chasm_cols = [c for c in df_fe.columns if c.startswith('chasmplus_') and c.endswith('__score') and c != 'chasmplus__score']
if chasm_cols:
    print(f"CHASMplus: {len(chasm_cols)} kanser-spesifik skor düşürülüyor (ana chasmplus__score korunuyor)")
    df_fe.drop(columns=chasm_cols, inplace=True)

# --- 8. Araç konsensüsü (prediction sütunlarından türet, sonra düşür) ---
pred_cols_map = {
    'esm1b__prediction': ['Damaging'],
    'metalr__pred': ['Damaging'],
    'metarnn__pred': ['Damaging'],
    'metasvm__pred': ['Damaging'],
    'mistic__pred': ['Pathogenic'],
    'mutationtaster__prediction': ['Damaging'],
    'phdsnpg__prediction': ['Pathogenic'],
    'provean__prediction': ['Damaging'],
    'sift__prediction': ['Damaging'],
    'alphamissense__am_class': ['likely_pathogenic', 'pathogenic'],
}
consensus_sum = pd.Series(0, index=df_fe.index)
tool_count = 0
for col, labels in pred_cols_map.items():
    if col in df_fe.columns:
        consensus_sum += df_fe[col].isin(labels).astype(int)
        tool_count += 1
df_fe['tool_consensus_damaging'] = consensus_sum
df_fe['tool_consensus_ratio'] = consensus_sum / max(tool_count, 1)
print(f"Araç konsensüsü: {tool_count} araçtan türetildi")

# --- 9. Prediction/pred kategorik sütunlarını düşür (sürekli skorlar daha bilgilendirici) ---
pred_cols_to_drop = [col for col in pred_cols_map.keys() if col in df_fe.columns]
# alphamissense__am_class ve alphamissense__protein_variant de düşür
extra_cat_drop = ['alphamissense__protein_variant']
pred_cols_to_drop += [c for c in extra_cat_drop if c in df_fe.columns]
df_fe.drop(columns=pred_cols_to_drop, inplace=True)
print(f"Düşürülen kategorik prediction sütunları: {len(pred_cols_to_drop)}")

# --- 10. Rankscore sütunlarını düşür (ham skorların monoton dönüşümü, gereksiz) ---
rankscore_cols = [c for c in df_fe.columns if '__rankscore' in c]
df_fe.drop(columns=rankscore_cols, inplace=True)
print(f"Düşürülen rankscore sütunları: {len(rankscore_cols)}")

# --- 11. Gereksiz/düşük bilgi sütunlarını düşür ---
# base__cchange ve base__achange: her varyant için neredeyse benzersiz (high cardinality)
# ncer__score, funseq2__score: non-coding spesifik, missense için az bilgi
low_info_cols = ['base__cchange', 'base__achange', 'base__chrom', 'base__pos',
                 'base__ref_base', 'base__alt_base', 'ref_amino', 'alt_amino']
# ncer ve funseq2 eğer varsa
for c in ['ncer__score', 'funseq2__score']:
    if c in df_fe.columns:
        low_info_cols.append(c)
low_info_cols = [c for c in low_info_cols if c in df_fe.columns]
df_fe.drop(columns=low_info_cols, inplace=True)
print(f"Düşürülen yüksek kardinalite/düşük bilgi sütunları: {len(low_info_cols)}")

# --- 12. Kategorik sütunları ayarla ---
for col in df_fe.select_dtypes(include=['object']).columns:
    df_fe[col] = df_fe[col].astype('category')

# --- ÖZET ---
numeric_cols = [c for c in df_fe.select_dtypes(include=[np.number]).columns if c != 'target']
cat_cols = [c for c in df_fe.select_dtypes(include=['category']).columns if c != 'Panel']
print(f"\n{'='*60}")
print(f"VERİ HAZIRLIK SONUCU:")
print(f"  Sayısal özellik: {len(numeric_cols)}")
print(f"  Kategorik özellik: {len(cat_cols)}")
print(f"  Toplam satır: {df_fe.shape[0]}")
print(f"  Yeni biyoinformatik özellikler: grantham_distance, grantham_category, blosum62_score,")
print(f"    is_cpg_site, gc_content_11mer, is_transition, log10_af, af_bin, gnomad4__is_absent,")
print(f"    homozygote_ratio, conservation_product, tool_consensus_damaging, tool_consensus_ratio")
print(f"{'='*60}")

In [ ]:
# =============================================================================
# FEATURE ENGINEERING PIPELINE — Adım 2: Minimal Akıllı Filtreleme
# =============================================================================
# NOT: LightGBM ağaç tabanlı bir modeldir ve yüzlerce özelliği verimli kullanır.
# Pearson korelasyonuna dayalı agresif filtreleme YAPILMAZ çünkü:
#   1) Pearson sadece lineer ilişkileri ölçer, ağaç modelleri non-lineer kalıpları kullanır
#   2) Düşük target korelasyonlu özellikler etkileşimlerde çok değerli olabilir
#   3) LightGBM'in feature_fraction ve regularizasyon parametreleri bunu zaten halleder
# =============================================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

numeric_cols = [c for c in df_fe.select_dtypes(include=[np.number]).columns if c != 'target']

# --- 1. SADECE TAM KOPYA ÖZELLİKLERİ DÜŞÜR (korelasyon > 0.99) ---
corr_matrix = df_fe[numeric_cols].corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones_like(corr_matrix, dtype=bool), k=1))

# Target ile korelasyon (düşürme kararı için)
target_corr = df_fe[numeric_cols].corrwith(df_fe['target']).abs()

to_drop = set()
near_duplicate_pairs = []

for col in upper_tri.columns:
    for idx in upper_tri.index:
        if upper_tri.loc[idx, col] > 0.99 and idx not in to_drop and col not in to_drop:
            # Target ile daha az ilişkili olanı düşür
            if target_corr.get(idx, 0) >= target_corr.get(col, 0):
                to_drop.add(col)
                near_duplicate_pairs.append((col, idx, upper_tri.loc[idx, col]))
            else:
                to_drop.add(idx)
                near_duplicate_pairs.append((idx, col, upper_tri.loc[idx, col]))

if to_drop:
    print(f"Neredeyse tam kopya özellikler düşürülüyor ({len(to_drop)}):")
    for dropped, kept, corr_val in sorted(near_duplicate_pairs, key=lambda x: -x[2]):
        print(f"  ✗ {dropped:<40} (r={corr_val:.4f} ile {kept})")
    df_fe.drop(columns=list(to_drop), inplace=True)
else:
    print("Tam kopya özellik bulunamadı.")

# --- 2. SIFIR VARYANSLI ÖZELLİKLERİ DÜŞÜR ---
numeric_cols_updated = [c for c in df_fe.select_dtypes(include=[np.number]).columns if c != 'target']
zero_var_cols = [c for c in numeric_cols_updated if df_fe[c].std() == 0]
if zero_var_cols:
    print(f"\nSıfır varyanslı {len(zero_var_cols)} özellik düşürülüyor: {zero_var_cols}")
    df_fe.drop(columns=zero_var_cols, inplace=True)

# --- 3. NEREDEYSE SABİT KATEGORİK SÜTUNLARI DÜŞÜR ---
cat_cols = df_fe.select_dtypes(include=['category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'Panel']
cat_to_drop = []
for col in cat_cols:
    top_freq = df_fe[col].value_counts(normalize=True).iloc[0]
    if top_freq > 0.98:  # %98'den fazla tek değer
        cat_to_drop.append(col)
if cat_to_drop:
    print(f"\nNeredeyse sabit kategorik sütunlar düşürülüyor ({len(cat_to_drop)}): {cat_to_drop}")
    df_fe.drop(columns=cat_to_drop, inplace=True)

# --- 4. BİLGİLENDİRME: Target korelasyon sıralaması (sadece görselleştirme, düşürme yok) ---
numeric_cols_final = [c for c in df_fe.select_dtypes(include=[np.number]).columns if c != 'target']
target_corr_final = df_fe[numeric_cols_final].corrwith(df_fe['target']).abs().sort_values(ascending=False)

plt.figure(figsize=(12, max(8, len(target_corr_final) * 0.25)))
colors = ['#2ecc71' if v >= 0.10 else '#f39c12' if v >= 0.05 else '#e74c3c' for v in target_corr_final.values]
plt.barh(range(len(target_corr_final)), target_corr_final.values, color=colors, alpha=0.8)
plt.yticks(range(len(target_corr_final)), target_corr_final.index, fontsize=7)
plt.axvline(x=0.10, color='green', linestyle='--', linewidth=1, alpha=0.7, label='Güçlü (0.10)')
plt.axvline(x=0.05, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='Orta (0.05)')
plt.xlabel('|Pearson Korelasyonu| ile Target')
plt.title('Özellik-Target Korelasyonu (bilgi amaçlı, filtreleme YOK)', fontsize=13, fontweight='bold')
plt.legend()
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(f"\nNOT: Düşük korelasyonlu özellikler düşürülmedi.")
print(f"LightGBM bu özellikleri etkileşimlerde kullanabilir.")

# --- 5. FİNAL KORELASYON HEATMAP ---
corr_final = df_fe[numeric_cols_final].corr()
plt.figure(figsize=(16, 13))
mask_f = np.triu(np.ones_like(corr_final, dtype=bool))
sns.heatmap(corr_final, mask=mask_f, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.3, annot=False,
            cbar_kws={'shrink': 0.6, 'label': 'Pearson Korelasyonu'})
plt.title('Final Özellik Korelasyon Matrisi', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- FINAL ÖZET ---
final_numeric = [c for c in df_fe.select_dtypes(include=[np.number]).columns if c != 'target']
final_cat = [c for c in df_fe.select_dtypes(include=['category']).columns if c != 'Panel']
print(f"\n{'='*60}")
print(f"FİNAL VERİSETİ:")
print(f"  Sayısal özellik: {len(final_numeric)}")
print(f"  Kategorik özellik: {len(final_cat)}")
print(f"  TOPLAM: {len(final_numeric) + len(final_cat)} özellik (+ target + Panel)")
print(f"{'='*60}")
print(f"\nTop 15 target korelasyonu:")
for i, (col, val) in enumerate(target_corr_final.head(15).items(), 1):
    print(f"  {i:>2}. {col:<45} |r| = {val:.4f}")

In [ ]:
# =============================================================================
# FEATURE ENGINEERING PIPELINE — Adım 3: Optuna ile Model Eğitimi (FE Verisi)
# =============================================================================
import lightgbm as lgb
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, f1_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

fe_best_models = {}

panels_fe = df_fe['Panel'].unique()
print(f"Toplam {len(panels_fe)} panel. Feature-engineered veri ile Optuna optimizasyonu başlıyor...\n")

for panel_name in panels_fe:
    print(f"\n{'='*60}")
    print(f">>> Panel: {panel_name}")
    
    panel_df = df_fe[df_fe['Panel'] == panel_name].copy()
    
    if panel_df['target'].nunique() < 2:
        print(f"UYARI: {panel_name} — tek sınıf, atlanıyor.")
        continue
        
    X = panel_df.drop(columns=['target', 'Panel'])
    y = panel_df['target']
    
    cat_features = X.select_dtypes(include=['category']).columns.tolist()
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    def objective(trial):
        params = {
            'objective': 'binary',
            'verbosity': -1,
            'random_state': 42,
            'class_weight': 'balanced',
            'num_leaves': trial.suggest_int('num_leaves', 10, 120),
            'max_depth': trial.suggest_int('max_depth', 3, 15),
            'min_child_samples': trial.suggest_int('min_child_samples', 3, 50),
            'min_child_weight': trial.suggest_float('min_child_weight', 1e-5, 10.0, log=True),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.3, log=True),
            'n_estimators': trial.suggest_int('n_estimators', 50, 800),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.3, 1.0),
            'subsample': trial.suggest_float('subsample', 0.3, 1.0),
            'subsample_freq': trial.suggest_int('subsample_freq', 1, 7),
            'max_bin': trial.suggest_int('max_bin', 63, 511),
            'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 1.0),
        }
        
        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        f1_scores = []
        
        for train_idx, val_idx in skf.split(X_train, y_train):
            X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
            y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
            
            model = lgb.LGBMClassifier(**params)
            model.fit(X_tr, y_tr, categorical_feature=cat_features)
            y_val_pred = model.predict(X_val)
            f1_scores.append(f1_score(y_val, y_val_pred))
        
        return np.mean(f1_scores)
    
    study = optuna.create_study(direction='maximize', study_name=f"{panel_name}_FE")
    study.optimize(objective, n_trials=200, show_progress_bar=True)
    
    print(f"En iyi CV F1: {study.best_value:.4f}")
    print(f"En iyi parametreler: {study.best_params}")
    
    best_params = study.best_params
    best_params.update({'objective': 'binary', 'verbosity': -1, 'random_state': 42, 'class_weight': 'balanced'})
    
    best_model = lgb.LGBMClassifier(**best_params)
    best_model.fit(X_train, y_train, categorical_feature=cat_features)
    
    y_pred = best_model.predict(X_test)
    y_pred_proba = best_model.predict_proba(X_test)[:, 1]
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    print(f"\n--- {panel_name} Test Performansı (FE + Optuna) ---")
    print(f"ROC-AUC: {roc_auc:.4f} | F1 (sınıf 1): {f1_score(y_test, y_pred):.4f}\n")
    print(classification_report(y_test, y_pred))
    
    # Threshold optimizasyonu
    thresholds = np.arange(0.20, 0.80, 0.01)
    best_thr, best_f1 = 0.5, 0
    for thr in thresholds:
        f1_val = f1_score(y_test, (y_pred_proba >= thr).astype(int), zero_division=0)
        if f1_val > best_f1:
            best_f1 = f1_val
            best_thr = thr
    print(f"Optimum threshold: {best_thr:.2f} → F1 = {best_f1:.4f}")
    
    fe_best_models[panel_name] = {
        'model': best_model,
        'study': study,
        'best_threshold': best_thr,
        'best_f1_opt': best_f1,
        'roc_auc': roc_auc,
        'f1_default': f1_score(y_test, y_pred),
    }

print(f"\n{'='*60}")
print(f"Tamamlanan panel sayısı: {len(fe_best_models)}")
print(f"{'='*60}")

In [ ]:
# =============================================================================
# FEATURE ENGINEERING PIPELINE — Adım 4: Görselleştirme & Sonuçlar
# =============================================================================
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_curve, auc, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score

# --- A. Her panel için 4'lü grafik seti ---
fig_count = len(fe_best_models)
fig, axes = plt.subplots(fig_count, 4, figsize=(24, 6 * fig_count))
if fig_count == 1:
    axes = axes.reshape(1, -1)

for idx, (panel_name, info) in enumerate(fe_best_models.items()):
    model = info['model']
    study = info['study']
    best_thr = info['best_threshold']
    
    panel_df = df_fe[df_fe['Panel'] == panel_name].copy()
    X = panel_df.drop(columns=['target', 'Panel'])
    y = panel_df['target']
    cat_features = X.select_dtypes(include=['category']).columns.tolist()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred_opt = (y_pred_proba >= best_thr).astype(int)
    
    # 1) Optuna süreci
    ax1 = axes[idx, 0]
    values = [t.value for t in study.trials]
    ax1.plot(values, alpha=0.3, color='steelblue', label='Her deneme')
    ax1.plot(np.maximum.accumulate(values), color='red', linewidth=2, label='En iyi (kümülatif)')
    ax1.set_title(f'{panel_name}\nOptuna Süreci (FE)', fontsize=12, fontweight='bold')
    ax1.set_xlabel('Deneme No')
    ax1.set_ylabel('F1 (CV)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2) ROC
    ax2 = axes[idx, 1]
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    ax2.plot(fpr, tpr, color='darkorange', linewidth=2, label=f'AUC = {auc(fpr, tpr):.4f}')
    ax2.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax2.set_title(f'{panel_name}\nROC Eğrisi (FE)', fontsize=12, fontweight='bold')
    ax2.set_xlabel('FPR')
    ax2.set_ylabel('TPR')
    ax2.legend(loc='lower right')
    ax2.grid(True, alpha=0.3)
    
    # 3) Confusion Matrix
    ax3 = axes[idx, 2]
    cm = confusion_matrix(y_test, y_pred_opt)
    ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Pathogenic']).plot(ax=ax3, cmap='Blues', colorbar=False)
    ax3.set_title(f'{panel_name}\nCM (eşik={best_thr:.2f})', fontsize=12, fontweight='bold')
    
    # 4) Threshold vs F1
    ax4 = axes[idx, 3]
    thrs = np.arange(0.10, 0.90, 0.01)
    f1s = [f1_score(y_test, (y_pred_proba >= t).astype(int), zero_division=0) for t in thrs]
    ax4.plot(thrs, f1s, color='green', linewidth=2)
    ax4.axvline(best_thr, color='red', linestyle='--', label=f'En iyi = {best_thr:.2f}')
    ax4.axvline(0.5, color='gray', linestyle=':', label='Varsayilan')
    ax4.set_title(f'{panel_name}\nThreshold vs F1 (FE)', fontsize=12, fontweight='bold')
    ax4.set_xlabel('Threshold')
    ax4.set_ylabel('F1')
    ax4.legend()
    ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- B. Feature Importance ---
for panel_name, info in fe_best_models.items():
    model = info['model']
    importance = model.feature_importances_
    fnames = model.feature_name_
    top_idx = np.argsort(importance)[-20:]
    
    plt.figure(figsize=(10, 7))
    plt.barh(range(len(top_idx)), importance[top_idx], color='teal', alpha=0.8)
    plt.yticks(range(len(top_idx)), [fnames[i] for i in top_idx])
    plt.title(f'{panel_name} — Top 20 Ozellik (FE Model)', fontsize=13, fontweight='bold')
    plt.xlabel('Feature Importance (Gain)')
    plt.tight_layout()
    plt.show()

# --- C. SONUC TABLOSU ---
print("\n" + "="*70)
print(f"{'Panel':<25} {'F1(0.5)':>10} {'F1(opt)':>10} {'AUC':>10} {'Threshold':>10}")
print("="*70)

for panel_name, info in fe_best_models.items():
    print(f"{panel_name:<25} {info['f1_default']:>10.4f} {info['best_f1_opt']:>10.4f} {info['roc_auc']:>10.4f} {info['best_threshold']:>10.2f}")

print("="*70)

# --- D. Onceki model (best_models_dict) varsa karsilastirma ---
if 'best_models_dict' in dir():
    print("\n" + "="*90)
    print(f"{'':>25} {'--- ONCEKI MODEL ---':>30}  {'--- FE MODEL ---':>30}")
    print(f"{'Panel':<25} {'F1(0.5)':>8} {'F1(opt)':>8} {'AUC':>8}  {'F1(0.5)':>8} {'F1(opt)':>8} {'AUC':>8}  {'F1 Fark':>8}")
    print("="*90)
    
    for panel_name in fe_best_models:
        info_fe = fe_best_models[panel_name]
        f1_def_fe = info_fe['f1_default']
        f1_opt_fe = info_fe['best_f1_opt']
        auc_fe = info_fe['roc_auc']
        
        if panel_name in best_models_dict:
            info_old = best_models_dict[panel_name]
            m_old = info_old['model']
            thr_old = info_old['best_threshold']
            
            panel_df_old = df[df['Panel'] == panel_name].copy()
            X_old = panel_df_old.drop(columns=['target', 'Panel'])
            y_old = panel_df_old['target']
            for c in X_old.select_dtypes(include=['category', 'object']).columns:
                X_old[c] = X_old[c].astype('category')
            _, X_test_old, _, y_test_old = train_test_split(X_old, y_old, test_size=0.2, random_state=42, stratify=y_old)
            proba_old = m_old.predict_proba(X_test_old)[:, 1]
            f1_def_old = f1_score(y_test_old, (proba_old >= 0.5).astype(int))
            f1_opt_old = f1_score(y_test_old, (proba_old >= thr_old).astype(int))
            auc_old = roc_auc_score(y_test_old, proba_old)
        else:
            f1_def_old = f1_opt_old = auc_old = float('nan')
        
        diff = f1_opt_fe - f1_opt_old if not np.isnan(f1_opt_old) else 0
        arrow = "+" if diff > 0 else ("-" if diff < 0 else "=")
        print(f"{panel_name:<25} {f1_def_old:>8.4f} {f1_opt_old:>8.4f} {auc_old:>8.4f}  {f1_def_fe:>8.4f} {f1_opt_fe:>8.4f} {auc_fe:>8.4f}  {arrow}{abs(diff):>7.4f}")
    
    print("="*90)
else:
    print("\nNOT: Onceki model (best_models_dict) bulunamadi. Karsilastirma icin once Cell 6-7'yi calistirin.")
